In [ ]:
from __future__ import annotations

In [ ]:
from collections.abc import Callable, Sequence, Iterator, Iterable
from dataclasses import dataclass
from itertools import product, chain, islice
from math import log10, exp, sqrt
from os import listdir
from pathlib import Path
from random import random, shuffle
from struct import unpack
from typing import cast, Final, NamedTuple, Self

In [ ]:
from numpy import array, float32, ndarray
from PIL.Image import Image, new, Resampling
from PIL.ImageDraw import Draw, ImageDraw
from tkinter import Button, Canvas, Event, Label, LEFT, RIGHT, StringVar, Tk

## Matrix

In [ ]:
class MatrixDim(NamedTuple):
    """Dimensions of a matrix."""

    m: int
    """Matrix height."""

    n: int
    """Matrix width."""

    @property
    def size(self) -> int:
        """The number of elements in the matrix."""

        return self.m * self.n

    @property
    def T(self) -> MatrixDim:
        """Dimensions of a transposed matrix."""

        return MatrixDim(m=self.n, n=self.m)

    @property
    def is_vector(self) -> bool:
        """Whether the matrix is a vector."""

        return self.m == 1 or self.n == 1

    def __str__(self) -> str:
        return f'{self.m} by {self.n}'

In [ ]:
@dataclass(slots=True)
class Matrix[Numeric: (bool, int, float)]:
    __values: tuple[Numeric, ...]
    __dim: MatrixDim
    __is_transposed: bool
    __render_decimals: int
    
    @property
    def dim(self) -> MatrixDim:
        """Matrix dimensions."""

        return self.__dim if not self.__is_transposed else self.__dim.T

    @property
    def T(self) -> Matrix[Numeric]:
        """Transposed matrix."""

        return Matrix(
            self.__values,
            dim=self.__dim,
            transposed=(not self.__is_transposed),
            render_decimals=self.__render_decimals,
        )

    @property
    def as_vector(self) -> Iterator[Numeric]:
        """Reshape the matrix to a vector."""

        if not self.dim.is_vector:
            raise ValueError(f'{self.dim} matrix is not a vector')
        return iter(self.__values)

    def apply[NewNumeric: (bool, float, int)](
        self, __func: Callable[[Numeric], NewNumeric]
    ) -> Matrix[NewNumeric]:
        """Apply a given transformation to each element of the matrix."""

        return Matrix(
            tuple(map(__func, self.__values)),
            dim=self.dim,
            transposed=False,
            render_decimals=self.__render_decimals,
        )

    def __init__(
        self,
        __values: tuple[Numeric, ...],
        /,
        dim: MatrixDim,
        *,
        transposed: bool = False,
        render_decimals: int = 0,
    ) -> None:
        self.__values = __values
        self.__dim = dim
        self.__is_transposed = transposed
        self.__render_decimals = render_decimals
        if len(__values) != dim.size:
            raise ValueError(
                f'Cannot interpret {len(__values)} values as {self.dim} matrix'
            )

    def __repr__(self) -> str:
        fmt: str = f'{(
            int(log10(max(map(abs, self.__values)) or 1))
        )}>.{self.__render_decimals}f'
        return f'[{'\n'.join(
            f'{' ' * (i != 0)}[{' '.join(
                f'{' ' * int(self[i, j] >= 0)}{self[i, j]:{fmt}}'
                for j in range(self.dim.n)
            )}]'
            for i in range(self.dim.m)
        )}]'

    def __len__(self) -> int:
        match self.dim:
            case (1, l) | (l, 1):
                return l
        raise ValueError(f'{self.dim} matrix has no length, use matrix.dim instead')

    def __getitem__(self, index: tuple[int, int], /) -> Numeric:
        return self.__values[(
            index[self.__is_transposed] * (
                self.dim.m if self.__is_transposed else self.dim.n
            ) + index[1 - self.__is_transposed]
        )]

    def __add__(self, __other: Matrix[Numeric], /) -> Matrix[Numeric]:
        if self.dim != __other.dim:
            raise ValueError(f'Cannot add {self.dim} and {__other.dim} matrices')
        return Matrix(
            tuple(
                self[i, j] + __other[i, j] for i, j
                in product(range(self.dim.m), range(self.dim.n))
            ),
            dim=self.dim,
            transposed=False,
            render_decimals=max(self.__render_decimals, __other.__render_decimals),
        )

    def __iadd__(self, __other: Matrix[Numeric]) -> Self:
        if self.dim != __other.dim:
            raise ValueError(f'Cannot add {self.dim} and {__other.dim} matrices')
        self.__values = cast(
            tuple[Numeric, ...],
            tuple(
                self[i, j] + __other[i, j] for i, j
                in product(range(self.dim.m), range(self.dim.n))
            ),
        )
        return self

    def __sub__(self, __other: Matrix[Numeric], /) -> Matrix[Numeric]:
        if self.dim != __other.dim:
            raise ValueError(f'Cannot subtract {self.dim} and {__other.dim} matrices')
        return Matrix(
            tuple(
                self[i, j] - __other[i, j] for i, j
                in product(range(self.dim.m), range(self.dim.n))
            ),
            dim=self.dim,
            transposed=False,
            render_decimals=max(self.__render_decimals, __other.__render_decimals),
        )

    def __isub__(self, __other: Matrix[Numeric]) -> Self:
        if self.dim != __other.dim:
            raise ValueError(f'Cannot subtract {self.dim} and {__other.dim} matrices')
        self.__values = cast(
            tuple[Numeric, ...],
            tuple(
                self[i, j] - __other[i, j] for i, j
                in product(range(self.dim.m), range(self.dim.n))
            ),
        )
        return self

    def __mul__(self, __other: Matrix[Numeric], /) -> Matrix[Numeric]:
        if self.dim != __other.dim:
            raise ValueError(
                f'Hadamard requires equal dims, got {self.dim} and {__other.dim}'
            )
        return Matrix(
            tuple(
                self[i, j] * __other[i, j] for i, j
                in product(range(self.dim.m), range(self.dim.n))
            ),
            dim=self.dim,
            render_decimals=self.__render_decimals,
        )

    def __imul__(self, __other: Matrix[Numeric], /) -> Self:
        if self.dim != __other.dim:
            raise ValueError(
                f'Hadamard requires equal dims, got {self.dim} and {__other.dim}'
            )
        self.__values = cast(
            tuple[Numeric, ...],
            tuple(
                self[i, j] * __other[i, j] for i, j
                in product(range(self.dim.m), range(self.dim.n))
            ),
        )
        return self

    def __matmul__(self, __other: Matrix[Numeric], /) -> Matrix[Numeric]:
        if self.dim.n != __other.dim.m:
            raise ValueError(f'Cannot multiply {self.dim} and {__other.dim} matrices')
        return Matrix(
            tuple(
                sum(self[i, k] * __other[k, j] for k in range(self.dim.n))
                for i, j in product(range(self.dim.m), range(__other.dim.n))
            ),
            dim=MatrixDim(m=self.dim.m, n=__other.dim.n),
            transposed=False,
            render_decimals=max(self.__render_decimals, __other.__render_decimals),
        )

    def __pow__(self, __power: int, /) -> Matrix[Numeric]:
        return self.apply(lambda x: x ** __power)  # type: ignore

    def __eq__(self, __other: object, /) -> bool:
        return isinstance(__other, Matrix) and self.dim == __other.dim and all(
            self[i, j] == __other[i, j] for i, j
            in product(range(self.dim.m), range(self.dim.n))
        )

    def __ne__(self, __other: object, /) -> bool:
        return not self == __other

In [ ]:
def matrix[Numeric: (bool, int, float)](
    __values: Sequence[Sequence[Numeric]], /, *, render_decimals: int = 0
) -> Matrix[Numeric]:
    """Construct a matrix from given values."""

    dim: MatrixDim = MatrixDim(
        m=len(__values), n=len(__values[0]) if len(__values) > 0 else 0
    )
    if not all(len(__values[i]) == dim.n for i in range(len(__values))):
        raise ValueError('Matrix rows have different lengths')
    return Matrix(tuple(chain(*__values)), dim, render_decimals=render_decimals)

In [ ]:
def vector[Numeric: (bool, int, float)](
    __values: Sequence[Numeric], /, *, render_decimals: int = 0
) -> Matrix[Numeric]:
    """Construct a vertical vector from given values."""

    return matrix([__values], render_decimals=render_decimals).T

In [ ]:
def zeros(dim: MatrixDim, *, render_decimals: int = 0) -> Matrix[float]:
    """Construct a zero matrix with given dimensions."""

    return Matrix((0.0,) * dim.size, dim=dim, render_decimals=render_decimals)

In [ ]:
def serialize_matrix[Numeric: (bool, int, float)](matrix: Matrix[Numeric]) -> str:
    """Convert a matrix to a string."""

    return f'{matrix.dim.m}*{matrix.dim.n}:{(
        ','.join(
            str(matrix[i, j]) for i, j in product(
                range(matrix.dim.m), range(matrix.dim.n)
            )
        )
    )}'

In [ ]:
def deserialize_matrix[Numeric: (bool, int, float)](
    string: str, dtype: type[Numeric] = float, render_decimals: int = 0
) -> Matrix[Numeric]:
    """Restore a matrix from the string."""

    dim_string, values_string = string.split(sep=':', maxsplit=1)
    dim: MatrixDim = MatrixDim(*map(int, dim_string.split(sep='*', maxsplit=1)))
    return Matrix(
        tuple(map(dtype, values_string.split(sep=','))),
        dim=dim,
        render_decimals=render_decimals,
    )

## Functions

In [ ]:
@dataclass(slots=True)
class Function[**Args, Ret]:
    __name: str
    __func: Callable[Args, Ret]
    __ddx_func: Callable[Args, Ret]

    @property
    def name(self) -> str:
        return self.__name

    @property
    def ddx(self) -> Callable[Args, Ret]:
        return self.__ddx_func

    def __init__(
        self,
        name: str,
        func: Callable[Args, Ret],
        ddx_func: Callable[Args, Ret],
    ) -> None:
        self.__name = name
        self.__func = func
        self.__ddx_func = ddx_func

    def __call__(self, x: Ret) -> Ret:
        return self.__func(x)

### Activation

In [ ]:
type ActivationFunction = Function[[float], float]
type LossFunction = Function[[Matrix[float], Matrix[float]], Matrix[float]]

In [ ]:
def sigmoid(x: float) -> float:
    return 1 / (1 + exp(-x))

In [ ]:
class Activation:
    ID: ActivationFunction = Function(
        name='Id',
        func=(lambda x: x),
        ddx_func=(lambda _: 1),
    )
    RELU: ActivationFunction = Function(
        name='ReLU',
        func=(lambda x: x * (x > 0)),
        ddx_func=(lambda x: x > 0),
    )
    SIGMOID: ActivationFunction = Function(
        name='Sigmoid',
        func=sigmoid,
        ddx_func=(lambda x: sigmoid(x) * (1 - sigmoid(x))),
    )

### Loss

In [ ]:
def softmax(v: Matrix[float]) -> Matrix[float]:
    exponents: list[float] = [exp(x - max(v.as_vector)) for x in v.as_vector]
    return vector([e / sum(exponents) for e in exponents])

In [ ]:
class Loss:
    MSE: LossFunction = Function(
        name='MSE',
        func=(lambda x, y: (y - x) ** 2),
        ddx_func=(lambda x, y: (x - y).apply(lambda v: 2 * v)),
    )
    SOFTMAX_CROSSENTROPY: LossFunction = Function(
        name='SoftmaxCrossentropy',
        func=(
            lambda z, y: (
                softmax(z) * y.apply(lambda t: -1.0 if t > 0 else 0.0)
            )
        ),
        ddx_func=(lambda z, y: softmax(z) - y),
    )

### Transform

In [ ]:
def argmax(matrix: Matrix[float]) -> int:
    return max(enumerate(matrix.as_vector), key=lambda element: element[1])[0]

## Layer

In [ ]:
@dataclass(slots=True)
class LinearLayer:
    __w: Matrix[float]
    __b: Matrix[float]
    __activation: ActivationFunction
    __last_input: Matrix[float] | None
    __last_z: Matrix[float] | None
    __grad_w: Matrix[float]
    __grad_b: Matrix[float]

    @property
    def weights(self) -> Matrix[float]:
        return self.__w

    @weights.setter
    def weights(self, w: Matrix[float], /) -> None:
        assert self.__w.dim == w.dim
        self.__w = w

    @property
    def biases(self) -> Matrix[float]:
        return self.__b

    @biases.setter
    def biases(self, b: Matrix[float], /) -> None:
        assert self.__b.dim == b.dim
        self.__b = b

    @property
    def activation(self) -> str:
        return self.__activation.name

    def accumulate_grads(self, upstream: Matrix[float]) -> Matrix[float]:
        if self.__last_input is None or self.__last_z is None:
            raise RuntimeError('feed_forward must be called before accumulate_grads')
        delta: Matrix[float] = upstream * self.__last_z.apply(self.__activation.ddx)
        self.__grad_w += (delta @ self.__last_input.T)
        self.__grad_b += delta
        return self.__w.T @ delta

    def feed_forward(self, input: Matrix[float]) -> Matrix[float]:
        self.__last_input = input
        self.__last_z = self.__w @ input + self.__b
        return self.__last_z.apply(self.__activation)

    def apply_grads(self, *, learning_rate: float, batch_size: int) -> None:
        scale: float = learning_rate / float(batch_size)
        self.__w -= self.__grad_w.apply(lambda g: scale * g)
        self.__b -= self.__grad_b.apply(lambda g: scale * g)
        self.__zero_gradients()

    def __init__(
        self,
        dim: MatrixDim,
        activation: ActivationFunction,
        render_decimals: int,
    ) -> None:
        limit: float = sqrt(6.0 / (dim.m + dim.n))
        self.__w = Matrix(
            tuple((2 * random() - 1) * limit for _ in range(dim.size)),
            dim=dim,
            render_decimals=2,
        )
        self.__b = vector(
            tuple(0.0 for _ in range(dim.m)),
            render_decimals=render_decimals,
        )
        self.__activation = activation
        self.__last_input = None
        self.__last_z = None
        self.__zero_gradients()

    def __len__(self) -> int:
        return len(self.__b)

    def __zero_gradients(self) -> None:
        self.__grad_w = zeros(self.__w.dim, render_decimals=0)
        self.__grad_b = zeros(self.__b.dim, render_decimals=0)

In [ ]:
@dataclass(frozen=True, slots=True)
class LayerDescription:
    n: int
    activation: ActivationFunction

In [ ]:
def layer(
    n: int, activation: ActivationFunction = Activation.ID
) -> LayerDescription:
    return LayerDescription(n=n, activation=activation)

## Batching

In [ ]:
def batches[T](
    iterable: Iterable[T], *, size: int, shuffled: bool = True
) -> Iterable[list[T]]:
    iterator: Iterator[T] = iter(iterable)
    while batch := list(islice(iterator, size)):
        if shuffled:
            shuffle(batch)
        yield batch

## Model

In [ ]:
@dataclass(slots=True)
class Model[Answer]:
    __name: str
    __input_size: int
    __layers: list[LinearLayer]
    __loss: LossFunction
    __transform: Callable[[Matrix[float]], Answer]

    @property
    def name(self) -> str:
        """Name of the model."""

        return self.__name

    @property
    def summary(self) -> str:
        """Summary of the model."""

        return f'Model: {self.__input_size} -> {(
            ' -> '.join(
                f'L({len(layer)}, {layer.activation})'
                for layer in self.__layers
            )
        )} -> {self.__transform.__name__}'

    def fit(
        self,
        xs: list[Matrix[float]],
        ys: list[Matrix[float]],
        *,
        batch_size: int = 16,
        epochs: int = 10,
        learning_rate: float = 0.01,
    ) -> None:
        """Train the model."""

        batch_count: int = (len(xs) + 1) // batch_size
        print('+-------+' + '-' * (batch_count + 2))
        print('| Epoch |' + f' {'Batches':^{(batch_count + 1)}} ')
        print('+-------+' + '-' * (batch_count + 2), end=str())
        for epoch in range(epochs):
            print(f'\n| {(f'{epoch + 1}/{epochs}'):^5} |', end=' ')
            for batch in batches(zip(xs, ys, strict=True), size=batch_size):
                print(end='#')
                self.__update_weights(batch, learning_rate=learning_rate)

    def evaluate(self, xs: Sequence[Matrix[float]], answers: Sequence[int]) -> float:
        """Evaluate model accuracy."""

        return sum(
            self.predict(x) == answer
            for x, answer in zip(xs, answers, strict=True)
        ) / len(xs)

    def predict(self, input: Matrix[float]) -> Answer:
        """Predict the answer on a given input."""

        return self.__transform(self.__feed_forward(input))

    def save(self, folder: str) -> None:
        """Save model weights & biases into a file."""

        with Path(f'{folder}/{self.__name}').open('w') as model_file:
            for layer in self.__layers:
                model_file.write(f'{serialize_matrix(layer.weights)}\n')
                model_file.write(f'{serialize_matrix(layer.biases)}\n')

    def load(self, folder: str) -> None:
        """Restore model weights & biases from the file."""

        with Path(f'{folder}/{self.__name}').open('r') as model_file:
            lines: list[str] = list(
                filter(lambda line: line != str(), model_file.readlines())
            )
            for i, (w, b) in enumerate(zip(lines[::2], lines[1::2])):
                self.__layers[i].weights = deserialize_matrix(w, dtype=float)
                self.__layers[i].biases = deserialize_matrix(b, dtype=float)

    def __init__(
        self,
        input_size: int,
        layers: list[LayerDescription],
        loss: LossFunction,
        transform: Callable[[Matrix[float]], Answer],
        *,
        name: str,
        render_decimals: int = 2,
    ) -> None:
        self.__name = name
        self.__input_size = input_size
        self.__layers = [
            LinearLayer(
                dim=MatrixDim(d.n, d_prev.n),
                activation=d.activation,
                render_decimals=render_decimals,
            )
            for d_prev, d in zip(
                chain([LayerDescription(input_size, Activation.ID)], layers), layers
            )
        ]
        self.__loss = loss
        self.__transform = transform

    def __repr__(self) -> str:
        return self.summary

    def __feed_forward(self, input: Matrix[float]) -> Matrix[float]:
        result: Matrix[float] = input
        for layer in self.__layers:
            result = layer.feed_forward(result)
        return result

    def __update_weights(
        self,
        batch: list[tuple[Matrix[float], Matrix[float]]],
        *,
        learning_rate: float = 0.01,
    ) -> None:
        if len(batch) == 0:
            return
        for x, y in batch:
            prediction: Matrix[float] = self.__feed_forward(x)
            upstream: Matrix[float] = self.__loss.ddx(prediction, y)
            for layer in reversed(self.__layers):
                upstream = layer.accumulate_grads(upstream)
        for layer in self.__layers:
            layer.apply_grads(learning_rate=learning_rate, batch_size=len(batch))

## Dataset

In [ ]:
def read_idx_images(path: Path) -> list[Matrix[float]]:
    with path.open('rb') as images_file:
        magic, num, rows, cols = cast(
            tuple[int, int, int, int],
            unpack('>IIII', images_file.read(16)),
        )
        if magic != 2051:
            raise ValueError(f'Not an IDX image file (magic {magic})')
        size: int = rows * cols
        data: bytes = images_file.read()
        if len(data) != num * size:
            raise ValueError('Unexpected file length for image data')

        images: list[Matrix[float]] = []
        mv: memoryview[int] = memoryview(data)
        for i in range(num):
            start: int = i * size
            vals: tuple[float, ...] = tuple(
                v / 255.0 for v in mv[start:(start + size)].tolist()
            )
            images.append(Matrix(vals, dim=MatrixDim(size, 1), render_decimals=2))
        return images

In [ ]:
def read_idx_labels(path: Path) -> list[int]:
    with path.open('rb') as labels_file:
        magic, num = cast(tuple[int, int], unpack('>II', labels_file.read(8)))
        if magic != 2049:
            raise ValueError(f'Not an IDX label file (magic {magic})')
        data = labels_file.read()
        if len(data) != num:
            raise ValueError('Unexpected file length for label data')
        mv: memoryview[int] = memoryview(data)
        return mv.tolist()

In [ ]:
def load_dataset(
    path: Path, prepare_image: Callable[[float], float] | None = None
) -> tuple[list[Matrix[float]], list[int], list[Matrix[float]], list[int]]:
    train_images: list[Matrix[float]] = [
        image.apply(prepare_image) if prepare_image is not None else image
        for image in read_idx_images(
            path / 'train-images-idx3-ubyte' / 'train-images-idx3-ubyte'
        )
    ]
    train_labels: list[int] = read_idx_labels(
        path / 'train-labels-idx1-ubyte' / 'train-labels-idx1-ubyte'
    )
    test_images: list[Matrix[float]] = [
        image.apply(prepare_image) if prepare_image is not None else image
        for image in read_idx_images(
            path / 't10k-images-idx3-ubyte' / 't10k-images-idx3-ubyte'
        )
    ]
    test_labels: list[int] = read_idx_labels(
        path / 't10k-labels-idx1-ubyte' / 't10k-labels-idx1-ubyte'
    )
    return (train_images, train_labels, test_images, test_labels)

In [ ]:
def one_hot(label: int, classes: int) -> Matrix[float]:
    """Make a 1-hot vertical vector with the specified number of classes."""

    vec: list[float] = [0.0] * classes
    vec[label] = 1
    return vector(vec)

## Run

In [ ]:
def choose_model(folder: str) -> tuple[bool, str]:
    models: dict[int, str] = {
        index: name for index, name in enumerate(listdir(folder))
    }
    prompt: str = 'Enter index to load or name to create a new model:\n' + '\n'.join(
        f'{index}: {name}' for index, name in models.items()
    )
    while True:
        user_input: str = input(prompt)
        try:
            return False, models[int(user_input)]
        except ValueError:
            return True, user_input
        except KeyError:
            print('Invalid index')

In [ ]:
train_new_model, model_name = choose_model(folder='models')

In [ ]:
x_train, y_train, x_test, y_test = load_dataset(path=Path('mnist'))

In [ ]:
model: Model[float] = Model(
    input_size=784,
    layers=[
        layer(n=16, activation=Activation.RELU),
        layer(n=16, activation=Activation.RELU),
        layer(n=10),
    ],
    loss=Loss.SOFTMAX_CROSSENTROPY,
    transform=argmax,
    name=model_name,
)

In [ ]:
model.summary

### Train

In [ ]:
if train_new_model:
    model.fit(
        xs=x_train,
        ys=[one_hot(y, classes=10) for y in y_train],
        batch_size=128,
        epochs=8,
        learning_rate=0.05,
    )
    model.save(folder='models')
else:
    model.load(folder='models')

## Drawing

In [ ]:
CANVAS_SIZE: Final[int] = 280
GRID_SIZE: Final[int] = 28
SCALE: Final[int] = CANVAS_SIZE // GRID_SIZE

In [ ]:
@dataclass(slots=True)
class DrawMnist:
    __model: Model[float]
    __root: Tk
    __canvas: Canvas
    __prediction_label: Label
    __prediction_label_var: StringVar
    __model_info_label: Label
    __model_info_label_var: StringVar
    __image: Image
    __draw: ImageDraw
    __clear_button: Button

    def start(self) -> None:
        """Start a live digit predictor."""

        self.__root.mainloop()

    def __init__(self, model: Model[float]) -> None:
        self.__model = model
        self.__root = Tk()
        self.__root.title('MNIST Live Prediction')
        self.__canvas = Canvas(
            self.__root, width=CANVAS_SIZE, height=CANVAS_SIZE, bg='black'
        )
        self.__canvas.pack(side=LEFT)
        self.__prediction_label_var = StringVar()
        self.__prediction_label = Label(
            self.__root, textvariable=self.__prediction_label_var, font=('Arial', 24)
        )
        self.__prediction_label.pack(side=RIGHT, padx=20)
        self.__model_info_label_var = StringVar()
        self.__model_info_label: Label = Label(
            self.__root,
            textvariable=self.__model_info_label_var,
            font=('Arial', 14),
            justify='left',
        )
        self.__model_info_label.pack(side=RIGHT, padx=20)
        self.__image = new('L', (CANVAS_SIZE, CANVAS_SIZE), 0)
        self.__draw = Draw(self.__image)
        self.__canvas.bind('<B1-Motion>', self.__paint)
        self.__canvas.bind('<B3-Motion>', self.__erase)
        self.__clear_button: Button = Button(
            self.__root,
            text='Clear',
            command=self.__clear,
            font=('Arial', 16),
        )
        self.__clear_button.pack(side=RIGHT, padx=20, pady=20)
        self.__update_model_info()
        self.__update_prediction()

    def __paint(self, event: Event) -> None:
        x1, y1 = (event.x - 8), (event.y - 8)
        x2, y2 = (event.x + 8), (event.y + 8)
        self.__canvas.create_oval(x1, y1, x2, y2, fill='white', outline='white')
        self.__draw.ellipse([x1, y1, x2, y2], fill=255)

    def __erase(self, event: Event) -> None:
        x1, y1 = (event.x - 8), (event.y - 8)
        x2, y2 = (event.x + 8), (event.y + 8)
        self.__canvas.create_oval(x1, y1, x2, y2, fill='black', outline='black')
        self.__draw.ellipse([x1, y1, x2, y2], fill=0)

    def __get_matrix(self) -> Matrix[float]:
        small_image: Image = self.__image.resize(
            (GRID_SIZE, GRID_SIZE), Resampling.LANCZOS
        )
        small_image_array: ndarray = array(small_image).astype(float32) / 255.0
        flat: list[float] = small_image_array.flatten().tolist()
        return Matrix(tuple(flat), dim=MatrixDim(GRID_SIZE * GRID_SIZE, 1))

    def __update_prediction(self) -> None:
        x: Matrix[float] = self.__get_matrix()
        prediction: float = self.__model.predict(x)
        self.__prediction_label_var.set(f'Prediction: {prediction}')
        self.__root.after(500, self.__update_prediction)

    def __update_model_info(self) -> None:
        name: str = self.__model.name
        accuracy: float = self.__model.evaluate(x_test[:1000], y_test[:1000])
        self.__model_info_label_var.set(
            f'Model: {name}\nAccuracy: {accuracy:.2%}'
        )

    def __clear(self) -> None:
        self.__canvas.delete('all')
        self.__draw.rectangle([(0, 0), (CANVAS_SIZE, CANVAS_SIZE)], fill=0)

In [ ]:
drawer: DrawMnist = DrawMnist(model)

In [ ]:
drawer.start()